In [1]:
# Import standard dependencies
import cv2
import os
import random
import uuid
import glob
import math
import numpy as np
from matplotlib import pyplot as plt

In [2]:
# Import PyTorch dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [4]:
# Setup paths
POS_PATH = os.path.join('data', 'positive')
NEG_PATH = os.path.join('data', 'negative')
ANC_PATH = os.path.join('data', 'anchor')

In [5]:

#os.makedirs(POS_PATH)
#os.makedirs(NEG_PATH)
#os.makedirs(ANC_PATH)

In [6]:
"""
for directory in os.listdir('lfw-deepfunneled/lfw-deepfunneled/'):
    for file in os.listdir(os.path.join('lfw-deepfunneled/lfw-deepfunneled/', directory)):
        EX_PATH = os.path.join('lfw-deepfunneled/lfw-deepfunneled/', directory, file)
        NEW_PATH = os.path.join(NEG_PATH, file)
        os.replace(EX_PATH, NEW_PATH)
"""

"\nfor directory in os.listdir('lfw-deepfunneled/lfw-deepfunneled/'):\n    for file in os.listdir(os.path.join('lfw-deepfunneled/lfw-deepfunneled/', directory)):\n        EX_PATH = os.path.join('lfw-deepfunneled/lfw-deepfunneled/', directory, file)\n        NEW_PATH = os.path.join(NEG_PATH, file)\n        os.replace(EX_PATH, NEW_PATH)\n"

In [7]:
#Establish the connection to webcam
'''
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    frame=frame[140:140+250, 150:150+250, :]

    #Collect Anchor Images
    if cv2.waitKey(1) & 0xFF == ord('a'):
        #Create a unique filename for the anchor image
        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)

    #Collect Positive Images
    if cv2.waitKey(1) & 0xFF == ord('p'):
        #Create a unique filename for the positive image
        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
        cv2.imwrite(imgname, frame)

    # Display the resulting frame
    cv2.imshow('Face Detection', frame)

    # Break the loop if 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
#Release the webcam and close the window
cap.release()
cv2.destroyAllWindows()
'''

"\ncap = cv2.VideoCapture(0)\nwhile cap.isOpened():\n    ret, frame = cap.read()\n    frame=frame[140:140+250, 150:150+250, :]\n\n    #Collect Anchor Images\n    if cv2.waitKey(1) & 0xFF == ord('a'):\n        #Create a unique filename for the anchor image\n        imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))\n        cv2.imwrite(imgname, frame)\n\n    #Collect Positive Images\n    if cv2.waitKey(1) & 0xFF == ord('p'):\n        #Create a unique filename for the positive image\n        imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))\n        cv2.imwrite(imgname, frame)\n\n    # Display the resulting frame\n    cv2.imshow('Face Detection', frame)\n\n    # Break the loop if 'q' is pressed\n    if cv2.waitKey(1) & 0xFF == ord('q'):\n        break\n#Release the webcam and close the window\ncap.release()\ncv2.destroyAllWindows()\n"

In [8]:
anchor = sorted(glob.glob(os.path.join(ANC_PATH, "*.jpg")))[:2000]
positive = sorted(glob.glob(os.path.join(POS_PATH, "*.jpg")))[:2000]
negative = sorted(glob.glob(os.path.join(NEG_PATH, "*.jpg")))[:2000]

In [9]:
# Data augmentation
data_augmentation = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(
        size=(100, 100),
        scale=(0.9, 1.0)
    ),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor()
])
test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((100, 100)), # Just resize, no random cropping
    transforms.ToTensor()
])

In [10]:
def preprocess(file_path, transform=None):
    byte_img = cv2.imread(file_path)
    img = cv2.cvtColor(byte_img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (100, 100))
    if transform:
        img = transform(img)
    return img


In [11]:
positives = list(zip(anchor, positive, [1.0] * len(anchor)))
negatives = list(zip(anchor, negative, [0.0] * len(anchor)))
data = positives + negatives

In [12]:
#Creating Data Pipeline
class SiameseDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        input_img_path, validation_img_path, label = self.data[index]

        return (
            preprocess(input_img_path, self.transform),
            preprocess(validation_img_path, self.transform),
            torch.tensor(label, dtype=torch.float32)
        )


In [13]:
random.shuffle(data)


In [14]:
train_size = round(len(data) * .7)
test_size = len(data) - train_size
train_data_raw, test_data_raw = random_split(data, [train_size, test_size])

train_dataset = SiameseDataset(train_data_raw, transform=data_augmentation)
test_dataset = SiameseDataset(test_data_raw, transform=test_transform)

train_data = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_data = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)


In [15]:
class Embedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=10)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=7)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(128, 128, kernel_size=4)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=4)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(256 * 5 * 5, 512)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = F.relu(self.conv3(x))
        x = self.pool3(x)
        x = F.relu(self.conv4(x))
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc1(x)
        return x


mod = Embedding()

In [16]:
# Siamese L1 Distance module
class L1Dist(nn.Module):

    # Init method - inheritance
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    # Magic happens here - similarity calculation
    def forward(self, input_embedding, validation_embedding):
        return torch.abs(input_embedding - validation_embedding)

In [17]:
l1=L1Dist()

In [18]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding()
        self.l1dist = L1Dist()
        self.classifier = nn.Linear(512, 1)

    def forward(self, input_image, validation_image):
        inp_embedding = self.embedding(input_image)
        val_embedding = self.embedding(validation_image)
        distances = self.l1dist(inp_embedding, val_embedding)
        return torch.sigmoid(self.classifier(distances))


siamese_network = SiameseNetwork().to(device)
print(siamese_network)

SiameseNetwork(
  (embedding): Embedding(
    (conv1): Conv2d(3, 64, kernel_size=(10, 10), stride=(1, 1))
    (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv2): Conv2d(64, 128, kernel_size=(7, 7), stride=(1, 1))
    (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv3): Conv2d(128, 128, kernel_size=(4, 4), stride=(1, 1))
    (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (conv4): Conv2d(128, 256, kernel_size=(4, 4), stride=(1, 1))
    (flatten): Flatten(start_dim=1, end_dim=-1)
    (dropout): Dropout(p=0.5, inplace=False)
    (fc1): Linear(in_features=6400, out_features=512, bias=True)
  )
  (l1dist): L1Dist()
  (classifier): Linear(in_features=512, out_features=1, bias=True)
)


In [19]:
binary_cross_loss = nn.BCELoss()
opt = torch.optim.Adam(siamese_network.parameters(), lr=1e-4)

In [20]:
checkpoint_dir = './training_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_prefix = os.path.join(checkpoint_dir, 'ckpt')

def save_checkpoint(path):
    torch.save({
        'model_state_dict': siamese_network.state_dict(),
        'optimizer_state_dict': opt.state_dict(),
    }, path)

def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    siamese_network.load_state_dict(ckpt['model_state_dict'])
    opt.load_state_dict(ckpt['optimizer_state_dict'])

In [21]:
def train_step(batch):

    # Get anchor and positive/negative image
    img_a, img_b = batch[0].to(device), batch[1].to(device)
    # Get label
    y = batch[2].to(device).unsqueeze(1)

    siamese_network.train()

    # Zero the gradients before the forward pass
    opt.zero_grad()

    # Forward pass
    yhat = siamese_network(img_a, img_b)
    # Calculate loss
    loss = binary_cross_loss(yhat, y)
    
    # Calculate gradients
    loss.backward()

    # Apply updated weights to the siamese model
    opt.step()

    # Return loss
    return loss, yhat


In [22]:
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score

def train(train_dataloader, val_dataloader, EPOCHS):
    for epoch in range(1, EPOCHS + 1):
        print(f"\nEpoch {epoch}/{EPOCHS}")

        all_labels = []
        all_preds = []

        # Loop through each batch
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch}/{EPOCHS} (Train)"):
            
            # Run train step
            loss, yhat = train_step(batch)

            all_labels.extend(batch[2].cpu().numpy().tolist())
            all_preds.extend((yhat.detach().cpu().numpy() > 0.5).astype(int).flatten().tolist())

        r = recall_score(all_labels, all_preds, zero_division=0)
        p = precision_score(all_labels, all_preds, zero_division=0)
        print(f"Train Loss: {loss.item():.4f}, Recall: {r:.4f}, Precision: {p:.4f}")

        # Validation loop
        val_labels = []
        val_preds = []
        siamese_network.eval()
        with torch.no_grad():
            for val_batch in tqdm(val_dataloader, desc=f"Epoch {epoch}/{EPOCHS} (Val)"):
                yhat_val = siamese_network(val_batch[0].to(device), val_batch[1].to(device))
                val_labels.extend(val_batch[2].cpu().numpy().tolist())
                val_preds.extend((yhat_val.cpu().numpy() > 0.5).astype(int).flatten().tolist())
        
        val_r = recall_score(val_labels, val_preds, zero_division=0)
        val_p = precision_score(val_labels, val_preds, zero_division=0)
        print(f"Validation Recall: {val_r:.4f}, Precision: {val_p:.4f}")

        if epoch % 10 == 0:
            save_checkpoint(checkpoint_prefix + f"-{epoch}.pt")


In [23]:
train(train_data, test_data, EPOCHS=50)



Epoch 1/50


Epoch 1/50 (Train): 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 175/175 [00:13<00:00, 13.17it/s]


Train Loss: 0.7834, Recall: 0.5979, Precision: 0.7342


Epoch 1/50 (Val): 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 75/75 [00:02<00:00, 29.95it/s]


Validation Recall: 0.6967, Precision: 0.9676

Epoch 2/50


...

Validation Recall: 0.9967, Precision: 1.0000

Epoch 50/50


Epoch 50/50 (Train): 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 175/175 [00:14<00:00, 11.90it/s]


Train Loss: 0.0001, Recall: 0.9986, Precision: 1.0000


Epoch 50/50 (Val): 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 75/75 [00:02<00:00, 25.55it/s]


Validation Recall: 1.0000, Precision: 1.0000


In [24]:
all_labels, all_preds = [], []

siamese_network.eval()
with torch.no_grad():
    for test_input, test_val, y_true in test_data:
        yhat = siamese_network(test_input.to(device), test_val.to(device)).cpu().numpy()
        all_labels.extend(y_true.numpy().tolist())
        all_preds.extend((yhat > 0.5).astype(int).flatten().tolist())

print(recall_score(all_labels, all_preds, zero_division=0),
      precision_score(all_labels, all_preds, zero_division=0))

1.0 1.0


In [ ]:
#torch.save(siamese_network.state_dict(), "siamese_modelv2.pth")

In [38]:
model=SiameseNetwork().to(device)
model.load_state_dict(torch.load("siamese_modelv2.pth",map_location=device))

<All keys matched successfully>

In [39]:
ANC_PATH = os.path.join('testing', 'input')
POS_PATH = os.path.join('testing', 'validation')

In [40]:
def preprocess(file_path, transform=None):
    byte_img = cv2.imread(file_path)
    img = cv2.cvtColor(byte_img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (100, 100))
    if transform:
        img = transform(img)
    return img


In [ ]:
def verify(model, ANC_PATH, POS_PATH, n=5, verification_threshold=0.5):
    model.eval()
    
    # Get an actual image file from the ANC_PATH directory
    input_filename = [f for f in os.listdir(ANC_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))][0]
    input_image_path = os.path.join(ANC_PATH, input_filename)
    input_img = preprocess(input_image_path, transform=test_transform).unsqueeze(0).to(device)
    
    folder_averages = {}
    
    # Iterate through folders inside validation path
    for folder_name in os.listdir(POS_PATH):
        folder_path = os.path.join(POS_PATH, folder_name)
        if not os.path.isdir(folder_path):
            continue
            
        results = []
        for val_filename in os.listdir(folder_path):
            val_img_path = os.path.join(folder_path, val_filename)
            if not val_filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                continue
            
            val_img = preprocess(val_img_path, transform=test_transform).unsqueeze(0).to(device)
            with torch.no_grad():
                yhat = model(input_img, val_img)
                results.append(yhat.item())
        
        if len(results) > 0:
            results.sort(reverse=True)
            top_n = results[:50]
            folder_averages[folder_name] = sum(top_n) / len(top_n)
            
    best_match = None
    highest_avg = 0
    
    if folder_averages:
        best_match = max(folder_averages, key=folder_averages.get)
        highest_avg = folder_averages[best_match]
        
    if highest_avg >= verification_threshold:
        verified_identity = best_match
    else:
        verified_identity = "Unverified"
        
    return verified_identity, highest_avg, folder_averages


In [45]:
a=verify(model, ANC_PATH, POS_PATH, verification_threshold=0.5)
scores=a[0]
a

('Lokesh',
 0.9997905850410461,
 {'Lokesh': 0.9997905850410461,
  'Sam': 3.6861175556346382e-09,
  'yash': 0.9977510571479797})

In [37]:
np.mean(scores)

np.float64(0.43936960599381986)